# Day 3 — Full Analysis, Figures & Paper Tables
**Paper:** Can Synthetic Data Replace Real Data? (SLTE Framework)

### What this notebook produces
- **Figure 1:** AUC-ROC degradation curve (No-SLTE) — U-shaped across mixing ratios
- **Figure 2:** SLTE vs No-SLTE AUC comparison (tree vs parametric side-by-side)
- **Figure 3:** LCS distribution + confidence track breakdown
- **Figure 4:** F1 degradation heatmap (all classifiers × all ratios)
- **Tables 1–6:** Paper-ready Excel workbook (accuracy, F1, AUC, deltas, degradation summary)
- **key_numbers.json:** Every number cited in the paper, verified

### Files needed (upload before running)
- `tstr_results.csv` (Day 1)
- `slte_results.csv` (Day 2)
- `lcs_distribution.csv` (Day 2)
- `baseline_results.json` (Day 1)
- `day2_summary_log.json` (Day 2)
---

In [ ]:
# ════════════════════════════════════════════════
# CELL 1 — Imports & Load
# ════════════════════════════════════════════════
!pip install -q matplotlib openpyxl seaborn

import pandas as pd
import numpy as np
import json, warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
warnings.filterwarnings('ignore')

# Load all results
df_no  = pd.read_csv('tstr_results.csv')       # Day 1
df_sl  = pd.read_csv('slte_results.csv')        # Day 2
lcs_df = pd.read_csv('lcs_distribution.csv')   # Day 2
with open('baseline_results.json') as f:
    baseline = json.load(f)
with open('day2_summary_log.json') as f:
    day2_log = json.load(f)

TREE    = ['Decision Tree','Random Forest','Gradient Boosting']
PARAM   = ['Logistic Regression','Naive Bayes']
ALL_CLF = TREE + PARAM
RATIOS  = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
RATIO_LABELS = ['0%\n(100% Real)','20%','40%','60%','80%','100%\n(100% Synth)']

COLORS = {
    'Decision Tree'       : '#D85A30',
    'Random Forest'       : '#BA7517',
    'Gradient Boosting'   : '#6B3FA0',
    'Logistic Regression' : '#185FA5',
    'Naive Bayes'         : '#1D9E75',
}

print('✅ All files loaded')
print(f'No-SLTE results: {df_no.shape}  |  SLTE results: {df_sl.shape}')
print(f'LCS samples: {len(lcs_df):,}')

In [ ]:
# ════════════════════════════════════════════════
# CELL 2 — FIGURE 1: AUC-ROC Degradation (No-SLTE)
# Primary finding: performance degrades as synthetic proportion increases
# ════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 5.5))

for clf in ALL_CLF:
    sub  = df_no[df_no['classifier']==clf].sort_values('synthetic_ratio')
    ls   = '-o' if clf in TREE else '--s'
    lw   = 2.2  if clf in TREE else 1.8
    ax.plot(sub['synthetic_ratio'], sub['auc_roc'], ls,
            label=clf, color=COLORS[clf], linewidth=lw, markersize=6)

# Shade the degradation zone
ax.axvspan(0.6, 1.0, alpha=0.06, color='red', label='_nolegend_')
ax.text(0.62, 0.575, 'Critical degradation\nzone (>60% synthetic)',
        fontsize=8, color='#A00000', style='italic')

ax.set_xlabel('Proportion of Synthetic Data in Training Set', fontsize=12)
ax.set_ylabel('AUC-ROC (on Fixed Real Test Set)', fontsize=12)
ax.set_title('Figure 1: AUC-ROC Degradation vs. Synthetic Data Proportion (No SLTE)', fontsize=13, fontweight='bold')
ax.set_xticks(RATIOS)
ax.set_xticklabels(RATIO_LABELS, fontsize=9)
ax.set_ylim(0.54, 0.78)
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)

# Annotate tree vs parametric at 100%
tree_vals  = [df_no[(df_no['classifier']==c)&(df_no['synthetic_ratio']==1.0)]['auc_roc'].values[0] for c in TREE]
param_vals = [df_no[(df_no['classifier']==c)&(df_no['synthetic_ratio']==1.0)]['auc_roc'].values[0] for c in PARAM]
ax.annotate(f'Tree avg: {np.mean(tree_vals):.3f}',
            xy=(1.0, np.mean(tree_vals)), xytext=(0.82, np.mean(tree_vals)+0.025),
            arrowprops=dict(arrowstyle='->', color='#8B2000'), color='#8B2000', fontsize=9)
ax.annotate(f'Param avg: {np.mean(param_vals):.3f}',
            xy=(1.0, np.mean(param_vals)), xytext=(0.72, np.mean(param_vals)-0.035),
            arrowprops=dict(arrowstyle='->', color='#0A3D6B'), color='#0A3D6B', fontsize=9)

plt.tight_layout()
plt.savefig('fig1_auc_degradation.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Figure 1 saved')

In [ ]:
# ════════════════════════════════════════════════
# CELL 3 — FIGURE 2: SLTE vs No-SLTE Comparison
# ════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, clfs, title in zip(axes, [TREE, PARAM], ['Tree-Based Classifiers', 'Parametric Classifiers']):
    for clf in clfs:
        sub_no = df_no[df_no['classifier']==clf].sort_values('synthetic_ratio')
        sub_sl = df_sl[df_sl['classifier']==clf].sort_values('synthetic_ratio')
        c = COLORS[clf]
        ax.plot(sub_no['synthetic_ratio'], sub_no['auc_roc'],
                '--o', color=c, linewidth=1.5, markersize=4, alpha=0.55)
        ax.plot(sub_sl['synthetic_ratio'], sub_sl['auc_roc'],
                '-s',  color=c, linewidth=2.2, markersize=5,
                label=clf.replace(' ',chr(10)) if len(clf)>17 else clf)
    ax.set_title(f'Figure 2: {title}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Synthetic Proportion', fontsize=10)
    ax.set_ylabel('AUC-ROC', fontsize=10)
    ax.set_xticks(RATIOS)
    ax.set_xticklabels(['0%','20%','40%','60%','80%','100%'], fontsize=8)
    ax.set_ylim(0.54, 0.78)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# Shared legend for line style
solid_line  = plt.Line2D([0],[0], color='gray', linewidth=2.2, linestyle='-',  label='With SLTE')
dashed_line = plt.Line2D([0],[0], color='gray', linewidth=1.5, linestyle='--', label='Without SLTE')
fig.legend(handles=[solid_line, dashed_line], loc='lower center',
           ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.04))

plt.suptitle('SLTE vs No-SLTE: AUC-ROC Across Synthetic Mixing Ratios',
             fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0,0.05,1,1])
plt.savefig('fig2_slte_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Figure 2 saved')

In [ ]:
# ════════════════════════════════════════════════
# CELL 4 — FIGURE 3: LCS Distribution
# ════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# 3a — LCS histogram
ax = axes[0]
n, bins, patches = ax.hist(lcs_df['lcs'], bins=45, edgecolor='white', linewidth=0.4)
low_thresh, high_thresh = 0.40, 0.70
for patch, left in zip(patches, bins[:-1]):
    if left < low_thresh:
        patch.set_facecolor('#D85A30')
    elif left < high_thresh:
        patch.set_facecolor('#BA7517')
    else:
        patch.set_facecolor('#1D9E75')
ax.axvline(low_thresh,  color='#8B2000', linestyle='--', linewidth=1.8, label=f'Low threshold ({low_thresh})')
ax.axvline(high_thresh, color='#0A5C3C', linestyle='--', linewidth=1.8, label=f'High threshold ({high_thresh})')
from matplotlib.patches import Patch
legend_patches = [
    Patch(color='#D85A30', label=f'LOW  (LCS<{low_thresh})'),
    Patch(color='#BA7517', label=f'MID  ({low_thresh}–{high_thresh})'),
    Patch(color='#1D9E75', label=f'HIGH (LCS≥{high_thresh})'),
]
ax.legend(handles=legend_patches, fontsize=9)
ax.set_xlabel('Label Confidence Score (LCS)', fontsize=11)
ax.set_ylabel('Number of Synthetic Samples', fontsize=11)
ax.set_title('Figure 3a: LCS Distribution across Synthetic Dataset', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.25, axis='y')

# 3b — Track counts bar
ax = axes[1]
track_counts = lcs_df['track'].value_counts()
tracks  = ['HIGH', 'MID', 'LOW']
counts  = [track_counts.get(t, 0) for t in tracks]
weights = ['w = 1.0', 'w = 0.7', 'w = 0.5']
colors  = ['#1D9E75', '#BA7517', '#D85A30']
bars = ax.bar([f'{t}\n({w})' for t,w in zip(tracks,weights)], counts,
              color=colors, edgecolor='white', width=0.55)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+300,
            f'{count:,}', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Figure 3b: SLTE Confidence Track Distribution', fontsize=11, fontweight='bold')
ax.set_ylabel('Number of Synthetic Samples', fontsize=11)
ax.grid(True, alpha=0.25, axis='y')
ax.set_ylim(0, max(counts)*1.15)

plt.tight_layout()
plt.savefig('fig3_lcs_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Figure 3 saved')

In [ ]:
# ════════════════════════════════════════════════
# CELL 5 — FIGURE 4: F1 Heatmap
# Shows F1 collapse clearly — best visual for imbalanced target
# ════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ratio_lbls = ['0%','20%','40%','60%','80%','100%']

for ax, df_use, title in zip(axes, [df_no, df_sl], ['No SLTE', 'With SLTE']):
    pivot = df_use.pivot_table(index='classifier', columns='synthetic_ratio', values='f1')
    pivot.columns = ratio_lbls
    pivot = pivot.loc[ALL_CLF]   # consistent ordering
    sns.heatmap(pivot, ax=ax, annot=True, fmt='.3f', cmap='RdYlGn',
                vmin=0.0, vmax=0.5, linewidths=0.5, cbar_kws={'label':'F1 Score'})
    ax.set_title(f'Figure 4: F1 Score Heatmap — {title}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Synthetic Data Proportion', fontsize=10)
    ax.set_ylabel('')
    ax.tick_params(axis='y', rotation=0)

plt.suptitle('F1 Score across Classifiers and Mixing Ratios', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_f1_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Figure 4 saved')

In [ ]:
# ════════════════════════════════════════════════
# CELL 6 — All Paper Tables → Excel Workbook
# ════════════════════════════════════════════════
ratio_lbls = ['0% (Real)','20%','40%','60%','80%','100% (Synth)']

with pd.ExcelWriter('Paper_Results_All_Tables.xlsx', engine='openpyxl') as writer:

    # Table 1 — AUC-ROC No-SLTE (PRIMARY METRIC)
    t = df_no.pivot_table(index='classifier',columns='synthetic_ratio',values='auc_roc').round(4)
    t.columns = ratio_lbls; t = t.loc[ALL_CLF]
    t.to_excel(writer, sheet_name='T1 AUC No-SLTE')

    # Table 2 — AUC-ROC With SLTE
    t = df_sl.pivot_table(index='classifier',columns='synthetic_ratio',values='auc_roc').round(4)
    t.columns = ratio_lbls; t = t.loc[ALL_CLF]
    t.to_excel(writer, sheet_name='T2 AUC SLTE')

    # Table 3 — AUC Delta (SLTE - No SLTE)
    t1 = df_no.pivot_table(index='classifier',columns='synthetic_ratio',values='auc_roc').loc[ALL_CLF]
    t2 = df_sl.pivot_table(index='classifier',columns='synthetic_ratio',values='auc_roc').loc[ALL_CLF]
    delta = (t2.values - t1.values).round(4)
    pd.DataFrame(delta, index=ALL_CLF, columns=ratio_lbls).to_excel(writer, sheet_name='T3 AUC Delta SLTE')

    # Table 4 — Accuracy No-SLTE
    t = df_no.pivot_table(index='classifier',columns='synthetic_ratio',values='accuracy').round(4)
    t.columns = ratio_lbls; t = t.loc[ALL_CLF]
    t.to_excel(writer, sheet_name='T4 Accuracy No-SLTE')

    # Table 5 — F1 No-SLTE
    t = df_no.pivot_table(index='classifier',columns='synthetic_ratio',values='f1').round(4)
    t.columns = ratio_lbls; t = t.loc[ALL_CLF]
    t.to_excel(writer, sheet_name='T5 F1 No-SLTE')

    # Table 6 — F1 With SLTE
    t = df_sl.pivot_table(index='classifier',columns='synthetic_ratio',values='f1').round(4)
    t.columns = ratio_lbls; t = t.loc[ALL_CLF]
    t.to_excel(writer, sheet_name='T6 F1 SLTE')

    # Table 7 — Full Degradation Summary
    rows = []
    for clf in ALL_CLF:
        base_auc = baseline[clf]['auc_roc']
        base_f1  = baseline[clf]['f1']
        fs_no_auc = df_no[(df_no['classifier']==clf)&(df_no['synthetic_ratio']==1.0)]['auc_roc'].values[0]
        fs_sl_auc = df_sl[(df_sl['classifier']==clf)&(df_sl['synthetic_ratio']==1.0)]['auc_roc'].values[0]
        fs_no_f1  = df_no[(df_no['classifier']==clf)&(df_no['synthetic_ratio']==1.0)]['f1'].values[0]
        fs_sl_f1  = df_sl[(df_sl['classifier']==clf)&(df_sl['synthetic_ratio']==1.0)]['f1'].values[0]
        rows.append({
            'Classifier'            : clf,
            'Type'                  : 'Tree' if clf in TREE else 'Parametric',
            'Baseline AUC'          : base_auc,
            'No-SLTE AUC @100%Synth': fs_no_auc,
            'SLTE AUC @100%Synth'   : fs_sl_auc,
            'AUC Drop No-SLTE (%)'  : round((base_auc-fs_no_auc)/base_auc*100, 2),
            'AUC Drop SLTE (%)'     : round((base_auc-fs_sl_auc)/base_auc*100, 2),
            'SLTE AUC Gain'         : round(fs_sl_auc - fs_no_auc, 4),
            'Baseline F1'           : base_f1,
            'No-SLTE F1 @100%Synth' : fs_no_f1,
            'SLTE F1 @100%Synth'    : fs_sl_f1,
            'F1 Drop No-SLTE (%)'   : round((base_f1-fs_no_f1)/base_f1*100, 2) if base_f1>0 else 0,
        })
    pd.DataFrame(rows).to_excel(writer, sheet_name='T7 Degradation Summary', index=False)

    # Table 8 — LCS track distribution
    lcs_df['track'].value_counts().rename_axis('Track').reset_index(
        name='Sample Count').to_excel(writer, sheet_name='T8 LCS Tracks', index=False)

    # Table 9 — Full raw results
    pd.concat([df_no.assign(condition='No-SLTE'),
               df_sl.assign(condition='SLTE')]).sort_values(
        ['synthetic_ratio','classifier']).to_excel(writer, sheet_name='Raw Results', index=False)

print('✅ Excel workbook saved: Paper_Results_All_Tables.xlsx (9 sheets)')

In [ ]:
# ════════════════════════════════════════════════
# CELL 7 — Key Numbers JSON (every number in paper)
# ════════════════════════════════════════════════
TREE  = ['Decision Tree','Random Forest','Gradient Boosting']
PARAM = ['Logistic Regression','Naive Bayes']

def get_auc(df, clf, ratio):
    return df[(df['classifier']==clf)&(df['synthetic_ratio']==ratio)]['auc_roc'].values[0]
def get_f1(df, clf, ratio):
    return df[(df['classifier']==clf)&(df['synthetic_ratio']==ratio)]['f1'].values[0]

tree_auc_drop_no_slte  = np.mean([(baseline[c]['auc_roc']-get_auc(df_no,c,1.0))/baseline[c]['auc_roc']*100 for c in TREE])
param_auc_drop_no_slte = np.mean([(baseline[c]['auc_roc']-get_auc(df_no,c,1.0))/baseline[c]['auc_roc']*100 for c in PARAM])
tree_auc_drop_slte     = np.mean([(baseline[c]['auc_roc']-get_auc(df_sl,c,1.0))/baseline[c]['auc_roc']*100 for c in TREE])
param_auc_drop_slte    = np.mean([(baseline[c]['auc_roc']-get_auc(df_sl,c,1.0))/baseline[c]['auc_roc']*100 for c in PARAM])
slte_gain_tree         = np.mean([get_auc(df_sl,c,1.0)-get_auc(df_no,c,1.0) for c in TREE])
slte_gain_param        = np.mean([get_auc(df_sl,c,1.0)-get_auc(df_no,c,1.0) for c in PARAM])

key_numbers = {
    # Dataset
    'dataset_total_records'         : 110527,
    'class_ratio'                   : '4:1 (No:Yes = 88208:22319)',
    'train_pool_size'               : 88420,
    'test_set_size'                 : 22106,
    'synthetic_samples_generated'   : 88420,
    'ctgan_epochs'                  : 300,
    # Baseline (100% real)
    'baseline_auc_logistic'         : baseline['Logistic Regression']['auc_roc'],
    'baseline_auc_naivebayes'       : baseline['Naive Bayes']['auc_roc'],
    'baseline_auc_decisiontree'     : baseline['Decision Tree']['auc_roc'],
    'baseline_auc_randomforest'     : baseline['Random Forest']['auc_roc'],
    'baseline_auc_gradientboosting' : baseline['Gradient Boosting']['auc_roc'],
    # Degradation No-SLTE
    'tree_avg_auc_drop_no_slte_pct' : round(tree_auc_drop_no_slte,  2),
    'param_avg_auc_drop_no_slte_pct': round(param_auc_drop_no_slte, 2),
    'rf_auc_drop_no_slte_pct'       : round((baseline['Random Forest']['auc_roc']-get_auc(df_no,'Random Forest',1.0))/baseline['Random Forest']['auc_roc']*100, 2),
    'dt_auc_drop_no_slte_pct'       : round((baseline['Decision Tree']['auc_roc']-get_auc(df_no,'Decision Tree',1.0))/baseline['Decision Tree']['auc_roc']*100, 2),
    'lr_auc_drop_no_slte_pct'       : round((baseline['Logistic Regression']['auc_roc']-get_auc(df_no,'Logistic Regression',1.0))/baseline['Logistic Regression']['auc_roc']*100, 2),
    # SLTE stats
    'lcs_mean'                      : day2_log['lcs_mean'],
    'low_confidence_samples'        : day2_log['low_confidence_count'],
    'mid_confidence_samples'        : day2_log['mid_confidence_count'],
    'high_confidence_samples'       : day2_log['high_confidence_count'],
    'labels_corrected_by_knn'       : day2_log['labels_corrected'],
    'label_correction_rate_pct'     : day2_log['correction_rate_pct'],
    # SLTE improvement
    'tree_avg_auc_drop_slte_pct'    : round(tree_auc_drop_slte,  2),
    'param_avg_auc_drop_slte_pct'   : round(param_auc_drop_slte, 2),
    'slte_avg_gain_tree_auc'        : round(float(slte_gain_tree),  4),
    'slte_avg_gain_param_auc'       : round(float(slte_gain_param), 4),
    # F1 drops
    'rf_f1_baseline'                : baseline['Random Forest']['f1'],
    'rf_f1_full_synth_no_slte'      : get_f1(df_no,'Random Forest',1.0),
    'rf_f1_full_synth_slte'         : get_f1(df_sl,'Random Forest',1.0),
    'lr_f1_baseline'                : baseline['Logistic Regression']['f1'],
    'lr_f1_full_synth_no_slte'      : get_f1(df_no,'Logistic Regression',1.0),
    'lr_f1_full_synth_slte'         : get_f1(df_sl,'Logistic Regression',1.0),
}

with open('key_numbers_for_paper.json','w') as f:
    json.dump(key_numbers, f, indent=2)

print('=== KEY NUMBERS FOR YOUR PAPER ===')
for k, v in key_numbers.items():
    print(f'  {k}: {v}')
print('\n✅ key_numbers_for_paper.json saved')

In [ ]:
# ════════════════════════════════════════════════
# CELL 8 — Print interpretation notes for paper writing
# ════════════════════════════════════════════════
print('='*65)
print('INTERPRETATION NOTES FOR YOUR PAPER (Day 4 writing)')
print('='*65)
print()
print('PRIMARY METRIC: Use AUC-ROC (not accuracy)')
print('  Reason: 4:1 class imbalance makes accuracy misleading.')
print('  Majority-class baseline accuracy = ~79.8%')
print('  AUC-ROC is threshold-independent and imbalance-robust.')
print()
print('FINDING 1 — Consistent AUC degradation as synthetic proportion rises')
print(f'  All 5 classifiers drop AUC when moving 0% → 100% synthetic')
print(f'  Tree avg drop  (no SLTE): {tree_auc_drop_no_slte:.2f}%')
print(f'  Param avg drop (no SLTE): {param_auc_drop_no_slte:.2f}%')
print()
print('FINDING 2 — F1 collapses severely for ensemble models')
print(f'  Random Forest F1: {baseline["Random Forest"]["f1"]:.4f} → {get_f1(df_no,"Random Forest",1.0):.4f} (drop={((baseline["Random Forest"]["f1"]-get_f1(df_no,"Random Forest",1.0))/baseline["Random Forest"]["f1"]*100):.1f}%)')
print(f'  Logistic Reg  F1: {baseline["Logistic Regression"]["f1"]:.4f} → {get_f1(df_no,"Logistic Regression",1.0):.4f} (most stable)')
print()
print('FINDING 3 — SLTE narrows the gap at high synthetic ratios')
print(f'  Avg AUC improvement (tree)  at 100% synth: {slte_gain_tree:+.4f}')
print(f'  Avg AUC improvement (param) at 100% synth: {slte_gain_param:+.4f}')
print()
print('SYNTHETIC LABEL CORRUPTION EVIDENCE')
print(f'  {day2_log["low_confidence_count"]:,} samples ({day2_log["low_confidence_count"]/88420*100:.1f}%) had LCS < 0.40')
print(f'  {day2_log["labels_corrected"]:,} labels corrected by kNN ({day2_log["correction_rate_pct"]}% of all synthetic)')
print(f'  This is your empirical evidence for synthetic label corruption')
print()
print('✅ Day 3 COMPLETE — all figures, tables, and key numbers ready for paper writing')

In [ ]:
# ════════════════════════════════════════════════
# CELL 9 — Download everything
# ════════════════════════════════════════════════
import shutil
shutil.make_archive('day3_all_outputs', 'zip', '.', '.')
from google.colab import files
files.download('day3_all_outputs.zip')
print('✅ All Day 3 outputs downloaded')
print()
print('Files in this zip you need for the paper:')
print('  fig1_auc_degradation.png      ← Figure 1')
print('  fig2_slte_comparison.png      ← Figure 2')
print('  fig3_lcs_distribution.png     ← Figure 3')
print('  fig4_f1_heatmap.png           ← Figure 4')
print('  Paper_Results_All_Tables.xlsx ← All tables (9 sheets)')
print('  key_numbers_for_paper.json    ← Every number cited in paper')